# MNIST 

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.layers import (
    BatchNormalization,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling2D,
    Input,
    MaxPooling2D,
    RandomBrightness,
    RandomContrast,
    RandomRotation,
    RandomZoom,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.datasets import mnist


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
(X_train , y_train) , (X_test, y_test) = mnist.load_data()

In [ ]:
print(f"shape X_train: {X_train.shape}")
print(f"shape X_test: {X_test.shape}")
print(f"shape y_train: {y_train.shape}")
print(f"shape y_test: {y_test.shape}")

In [ ]:
model = Sequential([
    # Input
    Input(shape=(28, 28, 1)),
    
    # Data Augmentation 
    RandomZoom(0.1),
    RandomBrightness(0.04),
    RandomContrast(0.06),
    RandomRotation(0.05),
    
    # Block one
    Conv2D(filters=32, kernel_size = (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size = (3,3), activation= "relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.2),
    
    
    # Block Two
    Conv2D(filters= 64, kernel_size=(3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size = (3,3), activation= "relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    Flatten(),
    
    # Classifier
    Dense(units=128, activation="relu"),
    Dropout(0.5),
    Dense(units=10, activation="softmax")
    
])

In [ ]:
print(model.summary())

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate = 0.01)

model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [ ]:
callbacks =[
    EarlyStopping(
        monitor= "val_loss",
        mode = "min",
        patience = 5,
        min_delta = 1e-4,
        restore_best_weights = True,
        start_from_epoch = 5,
        verbose= 1
    ),
    
    ReduceLROnPlateau(
        monitor = "val_loss",
        mode = "min",
        patience = 5,
        factor = 0.5,
        min_delta = 1e-4,
        min_lr = 1e-6,
        verbose=1 
    )
]

In [ ]:
X_train , X_val , y_train, y_val = train_test_split(X_train , y_train, test_size = 0.2, random_state = 42, stratify=y_train)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data = (X_val, y_val),
    epochs = 20,
    batch_size = 64,
    callbacks = callbacks,
    shuffle = True
)

In [ ]:
plt.figure()

plt.plot(history.history["loss"], label="Training Loss", linewidth=2, marker="o")
plt.plot(history.history["val_loss"], label=["Validation Loss"], linewidth = 2, c = "red", marker="s")
plt.legend()
plt.grid(True, alpha=0.3)

plt.title("CNN Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history["accuracy"], label="Accuracy", marker="o", linewidth=2, )
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", marker="s", linewidth=2, color="red")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["learning_rate"],
    label="Learning Rate",
    marker="o",
    linewidth=2,
)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy",
    marker="s",
    linewidth=2,
    color="red",
)

plt.xlabel("Training Epoch")
plt.ylabel("Metric Value")
plt.title("Learning Rate and Training Accuracy Across Epochs")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_prob = model.predict(X_test, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm,
            cmap="Blues",
            annot=True,
            fmt = "d",
            cbar=True,
            linewidths= 0.5)
plt.title("Lower triangular confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

In [ ]:
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
print(f"Accuracy score: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision score: {precision_score(y_test, y_pred, average="weighted"):.2f}")
print(f"F1 score score: {f1_score(y_test, y_pred, average="weighted"):.2f}")
print(f"Recall : {recall_score(y_test, y_pred, average="weighted"):.2f}")

In [ ]:
plt.figure(figsize=(20,16))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_test[i].squeeze(), cmap="gray")
    confidence = np.max(y_prob[i]) * 100
    plt.title(
        f"True {y_test[i]}\n Predicted: {y_pred[i]}({confidence:0.1f})%",
        fontsize=15
    )
    plt.axis("off")
    

plt.show()

In [ ]:
wrong_indices = np.where(y_test != y_pred)[0] # return (63, 0)
print(f"Total misclassified images: {len(wrong_indices)}")

In [ ]:
print(wrong_indices)

In [ ]:
plt.figure(figsize=(20, 20))

for i, idx in enumerate(wrong_indices[:25]):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    confidence = np.max(y_prob[idx]) * 100
    plt.title(
        f"True {y_test[idx]}\nPredicted: {y_pred[idx]} ({confidence:0.1f}%)"
    )
    plt.axis("off")
    
    
plt.tight_layout()
plt.show()

In [ ]:
mistakes = Counter(zip(y_test[wrong_indices], y_pred[wrong_indices]))
mistakes.most_common(10)

In [ ]:
wrong_confidences = np.max(y_prob[wrong_indices], axis=1)

top_wrong = wrong_indices[np.argsort(wrong_confidences)[-25:]]
top_wrong

In [ ]:
plt.figure(figsize=(20, 20))
for i, idx in enumerate(top_wrong):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    confidence = np.max(y_prob[idx]) * 100
    plt.title(
        f"True {y_test[idx]}\n Predicted {y_pred[idx]} ({confidence:0.1f})"
    )
    
plt.tight_layout()
plt.show()

In [ ]:
model = Sequential([
    # Input 
    Input(shape= (28 ,28 ,1)),
    
    # Data Augmentation 
    RandomBrightness(0.6),
    RandomContrast(0.5),
    RandomZoom(0.1),
    RandomRotation(0.5),
    
    # Block one
    
    Conv2D(filters= 32, kernel_size=(3,3), activation="relu", padding = "same"),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size = (3,3), activation= "relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(), # Default (pool_size = (2,2), stride= 1)
    Dropout(0.25),
    
    
    # Block two
    
    Conv2D(filters=64, kernel_size=(3,3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(filters=64,  kernel_size=(3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    GlobalAveragePooling2D(),
    
    Dense(units=128, activation="relu"),
    Dropout(0.5),
    Dense(units=10 ,activation="softmax")
    
])

In [ ]:
model.summary()

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate = 0.01)

model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [ ]:
callbacks =[
    EarlyStopping(
        monitor= "val_loss",
        mode = "min",
        patience = 5,
        min_delta = 1e-4,
        restore_best_weights = True,
        start_from_epoch = 5,
        verbose= 1
    ),
    
    ReduceLROnPlateau(
        monitor = "val_loss",
        mode = "min",
        patience = 5,
        factor = 0.5,
        min_delta = 1e-4,
        min_lr = 1e-6,
        verbose=1 
    )
]

In [ ]:
X_train , X_val , y_train, y_val = train_test_split(X_train , y_train, test_size = 0.2, random_state = 42, stratify=y_train)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data = (X_val, y_val),
    epochs = 20,
    batch_size = 64,
    callbacks = callbacks,
    shuffle = True
)

In [ ]:
plt.figure()

plt.plot(history.history["loss"], label="Training Loss", linewidth=2, marker="o")
plt.plot(history.history["val_loss"], label=["Validation Loss"], linewidth = 2, c = "red", marker="s")
plt.legend()
plt.grid(True, alpha=0.3)

plt.title("CNN Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history["accuracy"], label="Accuracy", marker="o", linewidth=2, )
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", marker="s", linewidth=2, color="red")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["learning_rate"],
    label="Learning Rate",
    marker="o",
    linewidth=2,
)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy",
    marker="s",
    linewidth=2,
    color="red",
)

plt.xlabel("Training Epoch")
plt.ylabel("Metric Value")
plt.title("Learning Rate and Training Accuracy Across Epochs")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_prob = model.predict(X_test, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm,
            cmap="Blues",
            annot=True,
            fmt = "d",
            cbar=True,
            linewidths= 0.5)
plt.title("Lower triangular confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

In [ ]:
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
print(f"Accuracy score: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision score: {precision_score(y_test, y_pred, average="weighted"):.2f}")
print(f"F1 score score: {f1_score(y_test, y_pred, average="weighted"):.2f}")
print(f"Recall : {recall_score(y_test, y_pred, average="weighted"):.2f}")

In [ ]:
plt.figure(figsize=(20,16))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_test[i].squeeze(), cmap="gray")
    confidence = np.max(y_prob[i]) * 100
    plt.title(
        f"True {y_test[i]}\n Predicted: {y_pred[i]}({confidence:0.1f})%",
        fontsize=15
    )
    plt.axis("off")
    

plt.show()

In [ ]:
wrong_indices = np.where(y_test != y_pred)[0] # return (63, 0)
print(f"Total misclassified images: {len(wrong_indices)}")

In [ ]:
plt.figure(figsize=(20, 20))

for i, idx in enumerate(wrong_indices[:25]):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    confidence = np.max(y_prob[idx]) * 100
    plt.title(
        f"True {y_test[idx]}\nPredicted: {y_pred[idx]} ({confidence:0.1f}%)"
    )
    plt.axis("off")
    
    
plt.tight_layout()
plt.show()

In [ ]:
mistakes = Counter(zip(y_test[wrong_indices], y_pred[wrong_indices]))
mistakes.most_common(10)

In [ ]:
wrong_confidences = np.max(y_prob[wrong_indices], axis=1)

top_wrong = wrong_indices[np.argsort(wrong_confidences)[-25:]]
top_wrong

In [ ]:
plt.figure(figsize=(20, 20))
for i, idx in enumerate(top_wrong):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    confidence = np.max(y_prob[idx]) * 100
    plt.title(
        f"True {y_test[idx]}\n Predicted {y_pred[idx]} ({confidence:0.1f})"
    )
    
plt.tight_layout()
plt.show()